## NEU 171L Lab 1 - Analysis of MRI data
Jupyter is an interactive, web-based python interface. Jupyter is a great tool for data-science, since you can iteratively develop your project and debug your code without having to continually reload data into memory. To run each cell of the jupyter notebook, use  $\texttt{ctrl+enter}$. In addition to jupyter, three python packages are used in this notebook: numpy (to perform computations + sampling), pandas (data format and statistic methods), and matplotlib (to plot the data) 


### Learning Objectives
- Use descriptive statistics (mean, standard deviation, standard error of the mean) to quantify the basic features of data
- Use a **bootstrap** to build the sampling distribution of a mean and relate its spread to the standard error of the mean
- Use a **permutation test** to ask whether a difference between two groups is larger than we would expect if the group labels were meaningless
- Understand what a **p-value** means with respect to the tested hypothesis


In [ ]:
# RUN THIS CELL to import python packages 

import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd

### Step 1: Load in and clean the data

In [ ]:
# Step 1: Load in + clean the data

# load in data for the entire population
data = (pd.read_csv("oasis_cross-sectional.csv")
        .dropna(axis=0, subset=["CDR", "nWBV"])) # drop the participants without a clinical dementia rating

# isolate participants with nonzero clinical dementia rating
dementia_data = data.loc[data.CDR > 0, :]

# isolate general population (healthy participants)
healthy_data = data.loc[data.CDR == 0.0, :]

#print some of the data
print("Sample of dementia data:")
print(dementia_data.head(10), "\n")

print("Sample of healthy data:")
print(healthy_data.head(10), "\n")


We are only seeing the first 10 rows in each data set (that is what the .head(10) function does). Just looking at the data, which group seems to have the higher brain volume (nWBV)?



### Step 2: Basic descriptive statistics

In this section you will summarize the normalized whole-brain volume (nWBV) of each group with three numbers: the **mean**, the **standard deviation (SD)**, and the **standard error of the mean (SEM)**.

The SEM measures how precisely we have estimated the *mean* — it shrinks as the sample gets larger, according to:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

where $N$ is the sample size. So before we can compute the SEM, we first need $N$ for each group.

**First, the sample size.** Run the cell below to count how many participants are in each group and save those counts as `n_healthy` and `n_dementia`. We will reuse these values later, so it is important to compute them now. The `.count()` method counts the number of values in a column.

In [ ]:
# Sample size of each group (this cell is already complete -- just run it).
# .count() counts the number of nWBV values in each group.
n_healthy   = healthy_data["nWBV"].count()
n_dementia  = dementia_data["nWBV"].count()

print("Sample size (N) of the healthy population:  %d" % n_healthy)
print("Sample size (N) of the dementia population: %d" % n_dementia)


**Next, the mean.** Calculate the **mean** normalized whole-brain volume (nWBV) for the healthy subjects. Apply the <code>mean</code> method to the column "nWBV" in the healthy_data and print the value. Note that the <code>.3f</code> in the print line tells Python to include only three places after the decimal point.

In [ ]:
# Mean of the healthy group (fill in the column name)
mean_healthy = healthy_data[...].mean()
print("Mean nWBV for the healthy population is %.3f" % (mean_healthy))


Calculate the **mean** normalized whole-brain volume for the dementia patients following the same pattern as above.

In [ ]:
# Mean of the dementia group
mean_dementia = ...
print(...)


**Next, the standard deviation.** Calculate and print the **standard deviation (SD)** of the nWBV for the healthy and dementia patients. The SD measures how spread out the individual values are. Method syntax is **.std()** -- copy the pattern you used for the mean, but replace "mean" with "std".

In [ ]:
# Standard deviation of each group (fill in using the .std() method)
sd_healthy  = healthy_data[...].std()
sd_dementia = ...

print("SD of nWBV for the healthy population is  %.3f" % (sd_healthy))
print(...)


**Finally, the standard error of the mean.** Now combine the SD and the sample size to get the SEM for the **healthy** group, using the equation from the top of this section:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

Two pieces of Python syntax you need here:
- **Square root:** `np.sqrt(x)` computes the square root of `x`.
- **Division:** the `/` operator divides one number by another.

You already have `sd_healthy` and `n_healthy` from the cells above -- use them to complete the calculation.

In [ ]:
# Standard error of the mean for the HEALTHY group.
# SEM = SD / sqrt(N)
# hint: put sd_healthy on top, and np.sqrt(n_healthy) on the bottom of the division.
sem_healthy = sd_healthy / np.sqrt(...)

print("SEM of nWBV for the healthy population is %.4f" % (sem_healthy))


In [ ]:
# Plot the data as a histogram (just run this cell)
fig, ax = plt.subplots(1, figsize=(10,8))
ax.hist(healthy_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='healthy_data')
ax.hist(dementia_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='dementia_data')

ax.set_title("Distribution of nWBV for Dementia vs Healthy Population")
ax.set_xlabel("nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend()

### Step 2b: Bootstrapping the sampling distribution of the mean

The SEM you just calculated with the formula estimates how much the *mean* would bounce around if you repeatedly re-sampled the healthy population. A **bootstrap** lets us see that variability directly instead of trusting a formula.

The idea: treat the healthy sample you have as a stand-in for the whole population, and draw many new "samples" from it **with replacement** (so the same participant can be picked more than once). Each new sample gives a new mean. Collecting thousands of these means shows us the **sampling distribution of the mean** -- and its spread should come out close to the SEM you computed with the formula.

**Your task:** the bootstrap only works if each resample is the **same size as the original healthy sample**. In the code below, fill in the `size=` argument with the value that represents that sample size. (You computed several sample-size-like numbers above -- `n_healthy`, `n_dementia`, `n_iter`. Only one of them is correct here. Think about which sample we are bootstrapping.)

Note the `replace=True`: this is what makes it a bootstrap. A permutation test (Step 3) uses `replace=False` instead, because it reshuffles a fixed set of labels rather than drawing new data.

In [ ]:
# Bootstrap the healthy group to build the sampling distribution of its mean.

n_iterations = 10000  # number of bootstrap resamples

bootstrap_means = np.zeros([n_iterations,])  # array to hold the mean of each resample

for i in range(n_iterations):

    # First: how big should each bootstrap resample be? It must be the SAME size
    # as the original healthy sample. Fill in the value below.
    # (n_healthy, n_dementia, and n_iterations are all in scope -- only one is right.)
    n_to_draw_each_resample = ...

    # Draw a bootstrap sample from the healthy group, WITH replacement.
    current_sample = np.random.choice(a=healthy_data["nWBV"],
                                      size=n_to_draw_each_resample,
                                      replace=True)

    # Save the mean of this resample.
    bootstrap_means[i] = current_sample.mean()

# --- Compare the bootstrap spread to the formula-based SEM ---
# The standard deviation of the bootstrap means IS an estimate of the SEM.
print("SEM from the formula (SD / sqrt(N)):      %.4f" % sem_healthy)
print("Std of the bootstrap means (should match): %.4f" % bootstrap_means.std())

# --- Plot the sampling distribution with SEM and 95% CI lines ---
# We use the formula-based sem_healthy from Step 2 for the lines below.
ci_low  = mean_healthy - 1.96 * sem_healthy
ci_high = mean_healthy + 1.96 * sem_healthy

fig, ax = plt.subplots(1, figsize=(10, 8))
ax.hist(bootstrap_means, rwidth=0.9, alpha=0.5, bins=30,
        label="Bootstrap distribution of the healthy mean")

# Mean
ax.axvline(mean_healthy, color="black", lw=2, label="Healthy mean")
# +/- 1 SEM
ax.axvline(mean_healthy - sem_healthy, color="C1", lw=2, ls="--", label="- 1 SEM")
ax.axvline(mean_healthy + sem_healthy, color="C1", lw=2, ls="--", label="+ 1 SEM")
# 95% CI (mean +/- 1.96 * SEM)
ax.axvline(ci_low,  color="C3", lw=2, ls=":", label="2.5% (95% CI lower)")
ax.axvline(ci_high, color="C3", lw=2, ls=":", label="97.5% (95% CI upper)")

ax.set_title("Bootstrap sampling distribution of the healthy mean nWBV")
ax.set_xlabel("Mean nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 3: Permutation test

We want to know whether healthy and dementia participants really differ in normalized whole-brain volume (nWBV), or whether the difference we see could easily have arisen by chance.

To do this we set up a **null hypothesis**: the "healthy" and "dementia" labels are arbitrary, and every participant's nWBV was drawn from the *same* underlying distribution. If that null hypothesis were true, then splitting the participants into a "healthy" group and a "dementia" group would be no different from shuffling the labels and dealing the participants into two random groups of the same sizes.

A **permutation test** builds the null distribution directly from this idea:

1. Compute the **observed statistic** — here, the difference in mean nWBV between the healthy and dementia groups as they actually are.
2. Pool all participants together, then **shuffle the labels** and re-split them into a random "healthy" group and a random "dementia" group of the same sizes as the real groups.
3. Recompute the difference in means for that shuffled split. This is one value from the null distribution.
4. Repeat many times (here, 3000) to build up the whole null distribution of differences we would expect *if the labels didn't matter*.

**Why shuffle *without* replacement?** Because we are re-assigning labels to a fixed set of real participants. Each participant appears exactly once in every shuffle — they just might land in the other group. This is different from a **bootstrap**, which samples *with* replacement because it is imitating the collection of brand-new data rather than relabeling the data we already have. Using `np.random.permutation` below reshuffles the pooled values without replacement, which is exactly the label-shuffling the null hypothesis describes.

Parts of the analysis code are already written. Fill in the sections marked with `...` to complete the analysis. You will interpret the results in the lab worksheet.

In [ ]:
# We will use a **for loop** to shuffle the labels and record the difference in means each time.
# We will shuffle 3000 times.
# Make sure the lines inside the **for loop** stay indented so they run all 3000 times.

n_iterations = 10000  # number of times we shuffle the labels and recompute the statistic

# Pool ALL participants' nWBV values together (healthy + dementia) into one array.
pooled_nWBV = pd.concat([healthy_data.nWBV, dementia_data.nWBV]).values

# The size of the dementia group. After each shuffle we will call the first
# n_dementia values the "dementia" group and the rest the "healthy" group.
n_dementia = dementia_data.nWBV.count()

# The OBSERVED statistic: the real difference in group means (healthy minus dementia).
# Fill in the two means (you can reuse mean_healthy and mean_dementia from Step 2).
observed_diff = ...  # hint: mean_healthy - mean_dementia
print("Observed difference in mean nWBV (healthy - dementia): %.4f" % observed_diff)

# Array to hold the difference in means from each shuffled (null) split.
perm_diffs = np.zeros([n_iterations,])

for i in range(n_iterations):

    # Shuffle all the pooled values WITHOUT replacement.
    # hint: np.random.permutation( **the pooled array** )
    shuffled = ...

    # Split the shuffled values into a random "dementia" group and "healthy" group.
    perm_dementia = shuffled[:n_dementia]   # first n_dementia values
    perm_healthy  = shuffled[n_dementia:]   # everything after that

    # Record the difference in means for this shuffled split (healthy - dementia).
    # hint: use .mean() on each group
    perm_diffs[i] = ...


In [ ]:
# Plot the NULL DISTRIBUTION: the differences in means produced by shuffling the labels.
# Each value in perm_diffs is (healthy mean - dementia mean) for one random reshuffle.
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.hist(perm_diffs, rwidth=0.9, alpha=0.5, bins=30,
        label="Null distribution of (healthy - dementia) mean differences\n(from shuffling the labels)")

# Emphasized vertical line at the REAL, measured difference between the two group means.
ax.axvline(observed_diff, color="C1", lw=3,
           label="Measured difference between healthy mean & dementia mean (healthy - dementia)")

ax.set_title("Permutation test: null distribution of (healthy - dementia) mean nWBV")
ax.set_xlabel("Difference in mean nWBV, healthy - dementia (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 4: P-value

The histogram you just made is the **null distribution**: all the (healthy - dementia) differences you would expect **if the labels didn't matter**. The emphasized line is the difference you **actually measured** in the real groups.

The **p-value** answers: *if the null hypothesis were true (labels meaningless), how often would chance alone produce a difference at least as extreme as the one we measured?* In a permutation test we read this straight off the null distribution -- it is the fraction of shuffled differences that land as far out as, or farther than, the measured line.

- A **small** p-value means the measured difference sits far out in the tail -- chance rarely produces something that extreme, so the difference is unlikely to be a fluke.
- A **large** p-value means the measured difference sits in the thick of the null distribution -- chance produces differences like it all the time, so we have little evidence the groups truly differ.

**This is a one-sided test.** We expect dementia to be associated with *lower* brain volume, which means a *positive* (healthy - dementia) difference. So we only count shuffles whose difference is **greater than or equal to** (`>=`) the measured difference -- i.e. as extreme *in that one direction*. (If we had no prior expectation about the direction, we would run a two-sided test and count extremes on both sides.)

Below you will first calculate the p-value the straightforward way, then look at a small correction statisticians recommend.

Note: "Boolean" refers to a list of values that are "True" or "False".

In [ ]:
# The operation for greater than or equal to is (>=) in python.
# Comparing the whole array to observed_diff gives a list of True/False values (Boolean),
# one for each shuffle: True where that shuffle was as extreme as (or more than) the measured difference.

# FILL IN: compare the shuffled differences to the measured difference.
boolean_more_extreme = ...   # hint: perm_diffs >= observed_diff

# FILL IN: .sum() counts how many values are True (how many shuffles were at least as extreme).
count_more_extreme = ...     # hint: boolean_more_extreme.sum()

print("%d out of %d shuffled differences were as extreme or more than the measured difference."
      % (count_more_extreme, n_iterations))

# FILL IN: the p-value is that count divided by the number of shuffles.
p_value = ...                # hint: count_more_extreme / n_iterations

print("p-value (uncorrected): %0.4f" % p_value)


#### A small correction to the p-value

Look at the number you just got. If **none** of your shuffles were as extreme as the measured difference, the formula above gives a p-value of exactly **0** -- which would claim it is *impossible* for chance to produce your result. But that can't be true: your real, unshuffled data is itself one possible arrangement of the labels, and it produced exactly the difference you measured. So chance producing that difference is clearly *not* impossible.

The fix is to **count the real data as one of the possible arrangements**. We add 1 to the numerator (the measured difference is always at least as extreme as itself) and a matching 1 to the denominator (it is one more arrangement in the total pool of shuffles + 1 real):

$$p = \frac{\text{count} + 1}{n_{\text{iterations}} + 1}$$

This is the p-value statisticians recommend for a permutation test. When the p-value is large the change is tiny, but at the small end it stops you from ever reporting an impossible p = 0. **Just run the cell below** -- you don't need to edit it.

In [ ]:
# Corrected permutation-test p-value: count the real data as one more arrangement.
# (Just run this cell -- nothing to fill in.)
p_value_corrected = (count_more_extreme + 1) / (n_iterations + 1)

print("p-value (uncorrected): %0.4f" % p_value)
print("p-value (corrected):   %0.4f" % p_value_corrected)


Answer the questions in the Lab 1 assignment to report and interpret the information in this notebook.